In [ ]:
# @title Installs
!pip install -U -q "google.genai"
!pip install PyMuPDF


In [ ]:
# @title 2. Setup and Imports
import google.generativeai as genai
import pydantic
from pydantic import BaseModel, Field, field_validator
from typing import List, Optional, Type
import pathlib
import json
import fitz  # PyMuPDF
import textwrap
import uuid
import os
from google.colab import userdata
import io # Added for in-memory byte streams
from google.generativeai import types # Added for constructing API requests
from google.genai import types
import google.generativeai as genai
import os
import time
import json
from pathlib import Path
from pydantic import BaseModel, Field
from tqdm import tqdm
import fitz  # PyMuPDF


try:
    os.environ["GEMINI_API_KEY"] = userdata.get("Muwaffaq_key_gemni")
    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
except userdata.SecretNotFoundError:
    print("ERROR: Please create a secret in Google Colab called 'GEMINI_API_KEY' and add your API key.")

In [ ]:
#@title Book Metadata
book_name = "Hizb Alba7r"

In [ ]:
# @title Spliting the Book
import fitz  # PyMuPDF library
import os

def split_pdf_into_pages(input_pdf_path):
    """
    Splits a multi-page PDF into an array of single-page PDFs.

    Args:
        input_pdf_path (str): The file path to the input PDF.

    Returns:
        list: A list of file paths to the newly created single-page PDFs.
              Returns None if the input file does not exist or an error occurs.
    """
    # Check if the input file exists
    if not os.path.exists(input_pdf_path):
        print(f"Error: The file at {input_pdf_path} was not found.")
        return None

    # Get the directory and base name for the output files
    input_dir, input_filename = os.path.split(input_pdf_path)
    base_filename, _ = os.path.splitext(input_filename)
    output_files = []

    try:
        # Open the input PDF document
        pdf_document = fitz.open(input_pdf_path)

        # Iterate through each page in the document
        for page_number in range(len(pdf_document)):
            # Create a new, single-page document
            single_page_doc = fitz.open()
            single_page_doc.insert_pdf(pdf_document, from_page=page_number, to_page=page_number)

            # Define the output filename for the single page
            output_filename = f"{base_filename}_page_{page_number + 1}.pdf"
            output_path = os.path.join(input_dir, output_filename)

            # Save the new single-page PDF
            single_page_doc.save(output_path)
            single_page_doc.close()

            print(f"Page {page_number + 1} saved to: {output_path}")
            output_files.append(output_path)

        return output_files

    except Exception as e:
        print(f"An error occurred: {e}")
        return None

In [ ]:

path_to_book = "/content/SAZELITEKKESI93.pdf"

# Call the function to split the PDF
print(f"\nSplitting PDF: {path_to_book}...")
page_files = split_pdf_into_pages(path_to_book)

if page_files:
    print("\nPDF splitting complete!")
    print(f"Found {len(page_files)} new files.")
    print(f"First 5 pages: {page_files[:5]}")
else:
    print("\nPDF splitting failed.")


Splitting PDF: /content/SAZELITEKKESI93.pdf...
Page 1 saved to: /content/SAZELITEKKESI93_page_1.pdf
Page 2 saved to: /content/SAZELITEKKESI93_page_2.pdf
Page 3 saved to: /content/SAZELITEKKESI93_page_3.pdf
Page 4 saved to: /content/SAZELITEKKESI93_page_4.pdf
Page 5 saved to: /content/SAZELITEKKESI93_page_5.pdf
Page 6 saved to: /content/SAZELITEKKESI93_page_6.pdf
Page 7 saved to: /content/SAZELITEKKESI93_page_7.pdf
Page 8 saved to: /content/SAZELITEKKESI93_page_8.pdf
Page 9 saved to: /content/SAZELITEKKESI93_page_9.pdf
Page 10 saved to: /content/SAZELITEKKESI93_page_10.pdf
Page 11 saved to: /content/SAZELITEKKESI93_page_11.pdf
Page 12 saved to: /content/SAZELITEKKESI93_page_12.pdf
Page 13 saved to: /content/SAZELITEKKESI93_page_13.pdf
Page 14 saved to: /content/SAZELITEKKESI93_page_14.pdf
Page 15 saved to: /content/SAZELITEKKESI93_page_15.pdf
Page 16 saved to: /content/SAZELITEKKESI93_page_16.pdf
Page 17 saved to: /content/SAZELITEKKESI93_page_17.pdf
Page 18 saved to: /content/SAZELITE

# Reading book (Manualy) —OLD

In [ ]:
# @title 1. Define the System Prompt for the Model
# This is the detailed instruction set for Gemini.
SYSTEM_PROMPT = """
You are an expert in Arabic Natural Language Processing (NLP) and Islamic literature. Your primary task is to meticulously process raw text extracted from scanned pages of Arabic or English books and convert it into a well-structured Markdown format inside a JSON. Your output must faithfully represent the original document's hierarchical structure and content layout.

For each document provided, perform the following steps:

1.  **Identify and Classify Text Blocks:** Analyze the provided text and classify each distinct section into one of the following four categories. The classification must be based on the content's role and placement on the page.

    -   **`main_text`**: The primary body of the text, typically organized into paragraphs.
    -   **`title`**: Prominent headings, including both main titles and any nested subtitles.
    -   **`marginalia`**: Annotations, notes, or commentary found in the margins of the page (known as *hawashi*).
    -   **`footnote`**: Explanatory notes or citations located at the bottom of the page.

2.  **Format the Output Using Structured Markdown:** Convert the classified text blocks into a single, cohesive Markdown output. Use the following specific formatting conventions to ensure consistency and clarity.

    -   **`title` and `subtitle`**: Use standard Markdown headings. A main title should be marked with `#` and subtitles with `##`, `###`, etc., to represent their hierarchy accurately.
    -   **`main_text`**: Present this text as standard paragraphs, ensuring all original paragraph breaks are preserved.
    -   **`marginalia`**: Prepend the text with a clearly defined tag to distinguish it from the main body. add `marginala` in the json followed by the text.
    -   **`footnote`**: Similarly, use a distinct tag for footnotes. add `footnote` in the json  followed by the text. (usually the footnotes comes after a divider in Arabic books, at the end of the page)

4. The user will send you the page number of the attached pages, you need to output the correct page number along with page content

**Example of the Expected JSON Output Format:**


{"page_content":"
# The Book's Main Title

## Chapter Subtitle

This is the main body of the text, consisting of one or more paragraphs. The flow and structure of the text should be preserved exactly as it appears in the original source.

Another paragraph of the main text.

"
},
"notes"{
  "footnote":This is a footnote providing a citation or additional detail.,
  "marginala": This is a note written in the margin. It might contain additional commentary or cross-references.
},
{
  "page_number":<Given by user>
}



Notes:
- When encountring empty page output {"page_content":"Empty Page",...}
- Remember to send the page number corretly from the user


"""






In [ ]:
# @title 2. Gemnin Main Function
import base64
import json
import os
from google import genai
from google.genai import types
from pydantic import BaseModel


class BookData(BaseModel):
    """Contains parsing results of the book"""
    page_content:str = Field(description="the Extracted page content in markdown string")
    page_number:int = Field(description="the page number of the extracted content")


def extract_text_gemini_no_stream(filepath, page_number: int,model = "gemini-2.5-flash"):
        client = genai.Client(api_key=os.environ.get("GEMINI_API_KEY"),)
        contents =[
              types.Content(
                  role="user",
                  parts=[
                      types.Part.from_bytes(
                    data=filepath.read_bytes(),
                    mime_type='application/pdf',
                  ),
                      types.Part.from_text(text=f"Extract the text in this page. Page number = {page_number}"),
                  ],
              ),
          ]


        # Generate content and then check usage_metadata from the final chunk
        generate_content_config = types.GenerateContentConfig(
            thinking_config=types.ThinkingConfig(
                thinking_budget=0,
                include_thoughts=False
            ),
           response_mime_type="application/json",
          response_schema=BookData,
            system_instruction=SYSTEM_PROMPT
        )

        full_answer_text = ""
        last_chunk = None
        print("\n--- AI STARTED! ---")

        response = client.models.generate_content(
            model=model,
            contents=contents,
            config=generate_content_config,
        )

        print("\n--- Generation Complete ---")
        print(f"Full answer text: {full_answer_text}")
        # Parse the answer text and add usage metadata
        markdown_text = markdown_text = json.loads(response.text)
        # Add usage as a standalone JSON object
        usage_data = {
                "prompt_token_count": response.usage_metadata.prompt_token_count,
                "candidates_token_count": response.usage_metadata.candidates_token_count,
                "total_token_count": response.usage_metadata.total_token_count,
                "thoughts_token_count" : response.usage_metadata.thoughts_token_count,
            }
        return {"result": markdown_text, "usage": usage_data}

In [ ]:
# @title 3. Saving JSON file of the results
version = f"{book_name}_Nawawi_version-01"
drive_folder_path="/content/drive/MyDrive/parsed_fiqh/Athkar_Nawawi"
file_path = f'{drive_folder_path}/{version}.json'

def process_and_save_results(gemini_result):
    """
    Saves a new Gemini result to a JSON file on Google Drive.

    This function reads existing data from the file, appends the new result,
    and saves the updated data back to the file. This ensures incremental updates
    without losing previous results.

    Args:
        gemini_result (dict): The complete JSON object returned from the Gemini API.
    """
    existing_data = []

    # 2. Check if the file exists and load its contents.
    # This prevents overwriting the file with each call.
    if os.path.exists(file_path):
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                existing_data = json.load(f)
            print(f"Loaded existing data from {file_path}")
        except (json.JSONDecodeError, FileNotFoundError) as e:
            print(f"Could not read or parse JSON file: {e}. Starting with a new file.")
            # If the file is empty or corrupted, we start with an empty list.

    # 3. Append the new result to the data list.
    existing_data.append(gemini_result)

    # 4. Save the updated data back to the file.
    # The 'w' mode will overwrite the file with the new, complete list.
    try:
        with open(file_path, 'w', encoding='utf-8') as f:
            json.dump(existing_data, f, indent=4, ensure_ascii=False)
        print(f"Successfully saved updated data to {file_path}")
    except IOError as e:
        print(f"Error saving file: {e}")

In [ ]:
# @title Getting the page number
import re

def extract_page_number(filename: str) -> int:
  """
  Extracts the page number from a filename with the format '_page_#.pdf'.

  Args:
    filename: The full filename as a string.

  Returns:
    The page number as an integer, or None if the pattern is not found.
  """
  try:
    # Use a regular expression to find the number between '_page_' and '.pdf'
    # The parentheses around `\d+` capture the number
    page_number = int(re.search(r'_page_(\d+)\.pdf', filename).group(1))
    return page_number
  except (AttributeError, ValueError):
    # This handles cases where the pattern is not found in the filename
    return None

In [ ]:
import time
import pathlib
import traceback
import math

# --- Start the timer ---
start_time = time.time()

# Your existing code
error_file_path = f'{drive_folder_path}/error__{version}.text'
all_results = []
max_retries = 3
retry_delay = 5

print("بسم الله!!!")
stack_trace = ""
for p in page_files[483:718]:
    attempt = 0
    success = False

    while attempt < max_retries and not success:
        try:
            print(f"Attempting to process page: {p} (Attempt {attempt + 1} of {max_retries})")
            filepath = pathlib.Path(p)
            page_number = extract_page_number(p)
            results = extract_text_gemini_no_stream(filepath, page_number)
            results["filename"] = p
            results["page_number"] = page_number
            process_and_save_results(results)
            all_results.append(results)
            print(f"Successfully processed {p}")
            success = True
            time.sleep(retry_delay)

        except Exception as e:
            stack_trace = traceback.format_exc()
            print("$Erorr$"*50)
            print(f"An error occurred while processing {p}. Error: {e}")
            print(f"Stack Trace:\n{stack_trace}")
            attempt += 1
            if attempt < max_retries:
                print(f"Retrying in {retry_delay} seconds...")
                time.sleep(retry_delay)

    if not success:
        print(f"Failed to process {p} after {max_retries} attempts. Logging error.")
        with open(error_file_path, 'a') as f:
            f.write(f"Failed to process {p} after {max_retries} attempts. Last error:{stack_trace}")
        break

print("-" * 30)
print("DONE!!!" * 100)

# --- End the timer and calculate duration ---
end_time = time.time()
duration_seconds = end_time - start_time

# Convert seconds to hh:mm format
hours = math.floor(duration_seconds / 3600)
minutes = math.floor((duration_seconds % 3600) / 60)

# Format the output string with leading zeros if needed
formatted_duration = f"{hours:02d}h:{minutes:02d}m"
print(f"Execution time: {formatted_duration}")

بسم الله!!!
------------------------------
DONE!!!DONE!!!DONE!!!DONE!!!DONE!!!DONE!!!DONE!!!DONE!!!DONE!!!DONE!!!DONE!!!DONE!!!DONE!!!DONE!!!DONE!!!DONE!!!DONE!!!DONE!!!DONE!!!DONE!!!DONE!!!DONE!!!DONE!!!DONE!!!DONE!!!DONE!!!DONE!!!DONE!!!DONE!!!DONE!!!DONE!!!DONE!!!DONE!!!DONE!!!DONE!!!DONE!!!DONE!!!DONE!!!DONE!!!DONE!!!DONE!!!DONE!!!DONE!!!DONE!!!DONE!!!DONE!!!DONE!!!DONE!!!DONE!!!DONE!!!DONE!!!DONE!!!DONE!!!DONE!!!DONE!!!DONE!!!DONE!!!DONE!!!DONE!!!DONE!!!DONE!!!DONE!!!DONE!!!DONE!!!DONE!!!DONE!!!DONE!!!DONE!!!DONE!!!DONE!!!DONE!!!DONE!!!DONE!!!DONE!!!DONE!!!DONE!!!DONE!!!DONE!!!DONE!!!DONE!!!DONE!!!DONE!!!DONE!!!DONE!!!DONE!!!DONE!!!DONE!!!DONE!!!DONE!!!DONE!!!DONE!!!DONE!!!DONE!!!DONE!!!DONE!!!DONE!!!DONE!!!DONE!!!DONE!!!DONE!!!
Execution time: 00h:00m


In [ ]:
with open("output.json", "w", encoding="utf-8") as f:
    json.dump(all_results, f, ensure_ascii=False, indent=4)

In [ ]:
for r in all_results[:20]:
  print(r["result"]["page_content"])
  print("\n --- \n")

# Parse by batch


```
`# I need to carefully check the code`
```
[Gemini pro Conversation](https://g.co/gemini/share/b2a2c87abfb1)


In [ ]:
from google import genai
client = genai.Client()

In [ ]:
# @title 1. Define the System Prompt for the Model
# This is the detailed instruction set for Gemini.
SYSTEM_PROMPT = """
You are an expert in Arabic Natural Language Processing (NLP) and Islamic literature. Your task is to process raw text from scanned pages of Arabic or English books, converting it into a structured **JSON** output. The JSON must contain the document's content formatted in Markdown, and it must accurately represent the original text's hierarchical structure and layout.

### Instructions

For each document, you'll be given the raw text and its corresponding page number. Your response must be a single JSON object.

1.  **Identify and Classify Text Blocks:** Analyze the provided text and categorize each distinct section into one of these four types:

      * **`main_text`**: The primary body of the text, typically in paragraphs.
      * **`title`**: Headings, including main titles (`#`) and nested subtitles (`##`, `###`).
      * **`marginalia`**: Annotations or commentary (*hawashi*) found in the page margins.
      * **`footnote`**: Explanatory notes or citations at the bottom of the page usualiy after a divider.

2.  **Format the Output:** Convert the classified text blocks into a single Markdown string under the `page_content` key within the JSON.

      * **Titles & Subtitles:** Use Markdown headings (`#`, `##`, etc.) to reflect the original hierarchy.
      * **Main Text:** Preserve paragraph breaks and present this as standard Markdown paragraphs.
      * **Marginalia & Footnotes:** Add these under the `notes` key in the JSON, with their respective keys `marginalia` and `footnote`.

3.  **Handle Page Numbers:** The user will provide the page number. Include this in the JSON output under the `page_number` key.

### Example JSON Output

```json
{
  "page_content": "# The Book's Main Title\n\n## Chapter Subtitle\n\nThis is the main body of the text, consisting of one or more paragraphs. The flow and structure of the text should be preserved exactly as it appears in the original source.\n\nAnother paragraph of the main text.",
  "notes": {
    "footnote": "This is a footnote providing a citation or additional detail.",
    "marginalia": "This is a note written in the margin. It might contain additional commentary or cross-references."
  },
  "page_number": "<Given by user>"
}
```

### Special Considerations

  * **Empty Pages:** If the provided text is an empty page, the `page_content` should be `"Empty Page"`.
  * **Accuracy:** Always output the correct page number as given by the user.

"""

In [ ]:
#  @title Json Schema
import typing

class Notes(BaseModel):
    """
    Represents the notes section of a book page, containing footnotes and marginalia.
    """
    footnote: Optional[str]
    marginalia: Optional[str]

class BookData(BaseModel):
    """
    Represents the structured content of a single book page.
    """
    page_content: str
    page_number: str
    notes: Notes

In [ ]:
# @title upload file
def uplod_file_to_google(pdf_files: List):
    print(f"\nUploading {len(pdf_files)} PDF files. This may take a while...")
    uploaded_files = {} # Dictionary to store {filepath: uploaded_file_object}
    for filepath in tqdm(pdf_files, desc="Uploading Files"):
        try:
            # We give the file a display name for easier identification if needed
            file_display_name = Path(filepath).name
            upload = upload_file_to_gemni_sdk(filepath,file_display_name)
            uploaded_files[filepath] = upload
            # A small delay to avoid hitting API rate limits on uploads
            time.sleep(0.5)
        except Exception as e:
            print(f"Failed to upload {filepath}: {e}")

    print(f"\n✅ Successfully uploaded {len(uploaded_files)} files.")
    return uploaded_files

def upload_file_to_gemni_sdk(path, display_name):
  uploaded_file = client.files.upload(
    file=path,
    config=types.UploadFileConfig(display_name=display_name, mime_type='application/pdf')
)
  return uploaded_file



In [ ]:
uploaded_files = uplod_file_to_google(page_files)


Uploading 63 PDF files. This may take a while...


Uploading Files: 100%|██████████| 63/63 [01:47<00:00,  1.70s/it]


✅ Successfully uploaded 63 files.


In [ ]:
#@title Build the jsonl file or inline src

def build_jsonl_for_file_for_gemnin3_sereise(system_instruction: str, user_prompt: str, page_number,file_uri):
    return {
        "key": f"request_{page_number}",
        "request": {
            "systemInstruction": { # Change to camelCase
                "parts": [{"text": system_instruction}]
            },
            "contents": [
                {
                    "role": "user",
                    "parts": [
                        {"text": user_prompt},
                        {
                            "fileData": { # Change to camelCase
                                "mimeType": "application/pdf", # Change to camelCase
                                "fileUri": file_uri # Change to camelCase
                            }
                        },
                        {"text": "JSON Schema: ..."}
                    ]
                }
            ],
            "generationConfig": { # Change to camelCase
                "responseMimeType": "application/json", # Change to camelCase
                "thinkingConfig": { # Change to camelCase
                    "thinkingLevel": "high", # Change to camelCase
                    "includeThoughts": True # Change to camelCase
                }
            }
        }
    }


def build_jsonl_for_file(system_instruction: str, user_prompt: str, page_number,file_uri):
  print("buliding JSONL")
  return {
  "key": f"request_{page_number}",
  "request": {
    "system_instruction": {
      "parts": [
        {
          "text": system_instruction
        }
      ]
    },
    "contents": [
      {
        "parts": [
          {
            "text": user_prompt
          },
          {
            "file_data": {
              "mime_type": "application/pdf",
              "file_uri": f"{file_uri}"
            }
          },
          {
            "text": "JSON Schema:\n{\n  \"type\": \"object\",\n  \"properties\": {\n    \"page_content\": {\"type\": \"string\"},\n    \"page_number\": {\"type\": \"string\"},\n    \"notes\": {\n      \"type\": \"object\",\n      \"properties\": {\n        \"footnote\": {\"type\": [\"string\", \"null\"]},\n        \"marginalia\": {\"type\": [\"string\", \"null\"]}\n      }\n    }\n  },\n  \"required\": [\"page_content\", \"page_number\", \"notes\"]\n}"
          }
        ]
      }
    ],
    "generation_config": {
      "response_mime_type": "application/json",
      "thinking_config":{
           "thinking_budget": -1
      }
    }
  }
}


def build_inline_request(system_instruction:str, user_prompt: str,file_uri):
  print("buliding Inline")
  return {
    "contents": [
      {
        "parts": [
          {
            "text": system_instruction
          },
          {
            "text": user_prompt
          },
          {
            "file_data": {
              "mime_type": "application/pdf",
              "file_uri": f"{file_uri}"
            }
          },
          {
            "text": "JSON Schema:\n{\n  \"type\": \"object\",\n  \"properties\": {\n    \"page_content\": {\"type\": \"string\"},\n    \"page_number\": {\"type\": \"string\"},\n    \"notes\": {\n      \"type\": \"object\",\n      \"properties\": {\n        \"footnote\": {\"type\": [\"string\", \"null\"]},\n        \"marginalia\": {\"type\": [\"string\", \"null\"]}\n      }\n    }\n  },\n  \"required\": [\"page_content\", \"page_number\", \"notes\"]\n}"
          }
        ]
      }
    ],
    "config": {
      "response_mime_type": "application/json",
      "thinking_config":{
           "thinking_budget": -1
      }
    }
    }

In [ ]:
# @title Create JsonL requests for the Batch
def creating_jsonL_for_batch(uploaded_files, inline=False):
  requests = []
  instruction_added =False
  for filepath, uploaded_file in tqdm(uploaded_files.items(), desc="Creating Requests"):
          page_number = int(Path(filepath).stem.split('_page_')[-1])
          user_prompt = f"Extract the text in this page. This is page number {page_number}."
          if inline:
            request = build_inline_request(SYSTEM_PROMPT,user_prompt,uploaded_file.uri)
          else:
            request = build_jsonl_for_file(SYSTEM_PROMPT, user_prompt,page_number, uploaded_file.uri)
            # request = build_jsonl_for_file_for_gemnin3_sereise(SYSTEM_PROMPT, user_prompt,page_number, uploaded_file.uri)
          requests.append(request)
  return requests

def creat_json_file(requests):
  json_file_path = f'{book_name}_batch_requests.jsonl'
  client = genai.Client()

  # Create a sample JSONL file
  with open(json_file_path, "w") as f:
      for req in requests:
          f.write(json.dumps(req) + "\n")

  # Upload the file to the File API
  uploaded_file = client.files.upload(
      file=json_file_path,
      config=types.UploadFileConfig(display_name=f'{book_name}_my-batch-requests-file', mime_type='jsonl')
  )
  return uploaded_file


In [ ]:
inline = False

In [ ]:
requests = creating_jsonL_for_batch(uploaded_files, inline)
print(requests)
if not inline:
  uploaded_batch_requests = creat_json_file(requests)
  print(uploaded_batch_requests)

Creating Requests: 100%|██████████| 63/63 [00:00<00:00, 9329.23it/s]


buliding JSONL
buliding JSONL
buliding JSONL
buliding JSONL
buliding JSONL
buliding JSONL
buliding JSONL
buliding JSONL
buliding JSONL
buliding JSONL
buliding JSONL
buliding JSONL
buliding JSONL
buliding JSONL
buliding JSONL
buliding JSONL
buliding JSONL
buliding JSONL
buliding JSONL
buliding JSONL
buliding JSONL
buliding JSONL
buliding JSONL
buliding JSONL
buliding JSONL
buliding JSONL
buliding JSONL
buliding JSONL
buliding JSONL
buliding JSONL
buliding JSONL
buliding JSONL
buliding JSONL
buliding JSONL
buliding JSONL
buliding JSONL
buliding JSONL
buliding JSONL
buliding JSONL
buliding JSONL
buliding JSONL
buliding JSONL
buliding JSONL
buliding JSONL
buliding JSONL
buliding JSONL
buliding JSONL
buliding JSONL
buliding JSONL
buliding JSONL
buliding JSONL
buliding JSONL
buliding JSONL
buliding JSONL
buliding JSONL
buliding JSONL
buliding JSONL
buliding JSONL
buliding JSONL
buliding JSONL
buliding JSONL
buliding JSONL
buliding JSONL
[{'key': 'request_1', 'request': {'system_instruction':

In [ ]:
uploaded_batch_requests

File(
  create_time=datetime.datetime(2026, 3, 13, 15, 20, 35, 322388, tzinfo=TzInfo(0)),
  display_name='Hizb Alba7r_my-batch-requests-file',
  expiration_time=datetime.datetime(2026, 3, 15, 15, 20, 35, 76568, tzinfo=TzInfo(0)),
  mime_type='jsonl',
  name='files/fe18wvls68jl',
  sha256_hash='ODZjNmQ5OWM1MTBkYmFjMzdmODIxYjBkMjFhZTU4NjhiNzA4MmYxYjI2OTgzOTJlNGYwMTAzMTVlMGZmMmU1ZA==',
  size_bytes=211662,
  source=<FileSource.UPLOADED: 'UPLOADED'>,
  state=<FileState.ACTIVE: 'ACTIVE'>,
  update_time=datetime.datetime(2026, 3, 13, 15, 20, 35, 322388, tzinfo=TzInfo(0)),
  uri='https://generativelanguage.googleapis.com/v1beta/files/fe18wvls68jl'
)

In [ ]:
# @title Starting the batch
file_batch_job = client.batches.create(
    model="gemini-2.5-pro",
    src= uploaded_batch_requests.name if not inline else requests,
    config={
        'display_name': f"{book_name}-job",
    },
)

print(f"Created batch job: {file_batch_job}")
print(f"Created batch job: {file_batch_job.name}")

Created batch job: name='batches/7ung57w2wnuh42ucd7pvvpr4hoj8a2vgy9bo' display_name='Hizb Alba7r-job' state=<JobState.JOB_STATE_PENDING: 'JOB_STATE_PENDING'> error=None create_time=datetime.datetime(2026, 3, 13, 15, 20, 38, 534133, tzinfo=TzInfo(0)) start_time=None end_time=None update_time=datetime.datetime(2026, 3, 13, 15, 20, 38, 534133, tzinfo=TzInfo(0)) model='models/gemini-2.5-pro' src=None dest=None completion_stats=None
Created batch job: batches/7ung57w2wnuh42ucd7pvvpr4hoj8a2vgy9bo


In [ ]:
# @title Determine Job name
job_name = "batches/7ung57w2wnuh42ucd7pvvpr4hoj8a2vgy9bo"

In [ ]:
# @title Monitor the Batch Status
batch_job = client.batches.get(name=job_name)

print(f"Polling status for job: {job_name}")
batch_job = client.batches.get(name=job_name) # Initial get
print(batch_job.display_name)
print(batch_job.state)


Polling status for job: batches/7ung57w2wnuh42ucd7pvvpr4hoj8a2vgy9bo
Hizb Alba7r-job
JobState.JOB_STATE_SUCCEEDED


In [ ]:
# @title Fetchig the Batch
# wait for the job to finish
import json
from google.colab import files
batch_job = client.batches.get(name=job_name)
json_result=[]

if batch_job.state.name == 'JOB_STATE_SUCCEEDED':

    # If batch job was created with a file
    if batch_job.dest and batch_job.dest.file_name:
        # Results are in a file
        result_file_name = batch_job.dest.file_name
        print(f"Results are in file: {result_file_name}")

        print("Downloading result file content...")
        file_content = client.files.download(file=result_file_name)
        # Process file_content (bytes) as needed
        print(file_content.decode('utf-8'))
        with open(f"{batch_job.display_name}_batch_result.jsonl", 'wb') as f:
            f.write(file_content)


    # If batch job was created with inline request
    # (for embeddings, use batch_job.dest.inlined_embed_content_responses)
    elif batch_job.dest and batch_job.dest.inlined_responses:
        # Results are inline
        print("Results are inline:")

        for i, inline_response in enumerate(batch_job.dest.inlined_responses):
            print(f"Response {i+1}:")
            if inline_response.response:
                # Accessing response, structure may vary.
                try:
                    print(inline_response.response.text)
                    print("######")
                    result = json.loads(inline_response.response.text)
                    print()
                    json_result.append(result)
                    print("######")
                except AttributeError:
                    print(inline_response.response) # Fallback
            elif inline_response.error:
                print(f"Error: {inline_response.error}")
    else:
        print("No results found (neither file nor inline).")
else:
    print(f"Job did not succeed. Final state: {batch_job.state.name}")
    if batch_job.error:
        print(f"Error: {batch_job.error}")


# Save json_result to a JSON file if inline is True
if inline:
    output_file_path = f"{batch_job.display_name}_batch_result.json"
    with open(output_file_path, "w", encoding="utf-8") as f:
        json.dump(json_result, f, ensure_ascii=False, indent=4)
    print(f"Inline results saved to {output_file_path}")

Results are in file: files/batch-7ung57w2wnuh42ucd7pvvpr4hoj8a2vgy9bo
{"key":"request_1","response":{"modelVersion":"gemini-2.5-pro","usageMetadata":{"candidatesTokenCount":158,"thoughtsTokenCount":1278,"promptTokensDetails":[{"tokenCount":698,"modality":"TEXT"},{"tokenCount":258,"modality":"DOCUMENT"}],"totalTokenCount":2392,"promptTokenCount":956},"responseId":"Vi60aYGjIo6UmtkPqtWRkQo","candidates":[{"content":{"role":"model","parts":[{"text":"{\n  \"page_content\": \"# كتاب الستر الأكبر والكبريت الأحمر\\n\\n## شرح حزب البحر\\n\\n### تأليف الشيخ البسطامي رحمه الله ورضي عنه، آمين.\\n\\nقد وقف هذا الكتاب الشريف إلى دركاه حضرت بن الشاذلي قد واقفر حضرت خديجة سلطان ابنة مصطفى خان ثلث قيادات وقفة لله.\",\n  \"notes\": {\n    \"footnote\": null,\n    \"marginalia\": \"شرح حزب البحر بسطامي\\n\\nخديجه بنت دلي على بن فا\\n\\nدرة عقد العصر في أسرار حزب البحر\"\n  },\n  \"page_number\": \"1\"\n}"}]},"finishReason":"STOP","index":0}]}}
{"key":"request_2","response":{"modelVersion":"gemini-2.5-pro

In [ ]:
# @title Read JSONL file
def read_jsonl_file(file_path):
    """
    Reads a JSONL file and prints each JSON object.

    Args:
        file_path (str): The path to the JSONL file.
    """
    results=[]
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            for line_number, line in enumerate(f, 1):
                try:
                    # Strip leading/trailing whitespace and newlines
                    json_object = json.loads(line.strip())
                    results.append(json_object)
                    # print(f"--- Line {line_number} ---")
                    # # Use json.dumps for pretty printing
                    # print(json.dumps(json_object, indent=4))
                    # print("-" * 20)
                except json.JSONDecodeError as e:
                    print(f"Error decoding JSON on line {line_number}: {e}")
                    print(f"Problematic line content: {line.strip()}")
    except FileNotFoundError:
        print(f"Error: The file at '{file_path}' was not found.")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
    return results

In [ ]:
if not inline:
    results = read_jsonl_file(f"{batch_job.display_name}_batch_result.jsonl")
    print(results[0])

{'key': 'request_1', 'response': {'modelVersion': 'gemini-2.5-pro', 'usageMetadata': {'candidatesTokenCount': 158, 'thoughtsTokenCount': 1278, 'promptTokensDetails': [{'tokenCount': 698, 'modality': 'TEXT'}, {'tokenCount': 258, 'modality': 'DOCUMENT'}], 'totalTokenCount': 2392, 'promptTokenCount': 956}, 'responseId': 'Vi60aYGjIo6UmtkPqtWRkQo', 'candidates': [{'content': {'role': 'model', 'parts': [{'text': '{\n  "page_content": "# كتاب الستر الأكبر والكبريت الأحمر\\n\\n## شرح حزب البحر\\n\\n### تأليف الشيخ البسطامي رحمه الله ورضي عنه، آمين.\\n\\nقد وقف هذا الكتاب الشريف إلى دركاه حضرت بن الشاذلي قد واقفر حضرت خديجة سلطان ابنة مصطفى خان ثلث قيادات وقفة لله.",\n  "notes": {\n    "footnote": null,\n    "marginalia": "شرح حزب البحر بسطامي\\n\\nخديجه بنت دلي على بن فا\\n\\nدرة عقد العصر في أسرار حزب البحر"\n  },\n  "page_number": "1"\n}'}]}, 'finishReason': 'STOP', 'index': 0}]}}


In [ ]:
# @title Write results to json file
import json
if not inline:
  output_file_path = f"{batch_job.display_name}_batch_result.json"

  with open(output_file_path, "w", encoding="utf-8") as f:
      json.dump(results, f, ensure_ascii=False, indent=4)

  print(f"Results saved to {output_file_path}")

Results saved to Hizb Alba7r-job_batch_result.json


In [ ]:
import json
import os

input_file = '/content/Hizb Alba7r-job_batch_result.json'
output_md = 'Hizb_Alba7r_Full_Book.md'

def extract_and_compile(json_path, output_path):
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    pages_data = []

    for item in data:
        try:
            # Navigate to the text content within the nested structure
            raw_text_json = item['response']['candidates'][0]['content']['parts'][0]['text']
            # Parse the inner JSON string containing page_content and page_number
            page_info = json.loads(raw_text_json)

            content = page_info.get('page_content', '')
            # Ensure page_number is an integer for correct sorting
            page_num = int(page_info.get('page_number', 0))

            pages_data.append({'number': page_num, 'content': content})
        except (KeyError, IndexError, json.JSONDecodeError, ValueError) as e:
            print(f"Skipping a malformed entry: {e}")

    # Sort pages by page number
    pages_data.sort(key=lambda x: x['number'])

    # Write to Markdown file
    with open(output_path, 'w', encoding='utf-8') as f_out:
        for page in pages_data:
            f_out.write(f"\n\n<!-- Page {page['number']} -->\n\n")
            f_out.write(page['content'])
            f_out.write("\n\n---\n") # Page divider

    print(f"Successfully created: {output_path}")

extract_and_compile(input_file, output_md)

Successfully created: Hizb_Alba7r_Full_Book.md
